In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


🥇 GOLD LAYER – Aggregation & KPIs
🔸 Enrich Orders with Returns & Inventory

In [ ]:
from pyspark.sql.functions import *


In [ ]:
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")


In [ ]:
display(df_returns)
display(df_orders)

In [ ]:
# 🟡 STEP 1: Join Orders with Returns (LEFT JOIN to retain all orders)
join_condition = df_orders.Order_ID == df_returns.OrderID
df_order_return = df_orders.join(df_returns, join_condition, how="left")
display(df_order_return)

In [ ]:
# 🟡 STEP 2: Join Inventory (LEFT JOIN on cleaned Product_Name)
df_enriched = df_order_return.join(
    df_inventory,
    df_order_return.Product_Name == df_inventory.ProductName,
    "left"
)
display(df_enriched)

In [ ]:
# 🟡 STEP 3: KPI Aggregations at Product Level
from pyspark.sql.functions import *

df_kpi = (
    df_enriched.groupBy("ProductName")
    .agg(
        count("OrderID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        count("ReturnID").alias("Total_Returns"),
        round((count("ReturnID") / count("OrderID")) * 100, 2).alias("Return_Rate"),
        round(sum("Order_Amount"), 2).alias("Total_Revenue"),
        round(avg("Order_Amount"), 2).alias("Avg_Order_Value"),
        sum("Stock").alias("Total_Stock"),
        round(avg("CostPrice"), 2).alias("Avg_Cost"),
        round(sum("Order_Amount") - (sum("Stock") * avg("CostPrice")), 2).alias("Net_Profit")
    )
)
display(df_kpi)

In [ ]:
# 🟡 STEP 4: Save to Gold Table
df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_kpis")